# 04 — Dissolution Model Explorer

Four mathematical models describe drug release from coated particles placed in dissolution medium.
This notebook lets you **explore model behaviour** by adjusting parameters — no experimental data needed.

| Model | Equation | Key parameter |
|---|---|---|
| Zero-order | F = k·t | k = release rate [%/min] |
| First-order | F = 100·(1 − e^(−k·t)) | k = rate constant [1/min] |
| Higuchi | F = k·√t | k = diffusion coefficient [%/min^0.5] |
| Korsmeyer-Peppas | F = 100·k·t^n | k = rate, n = release exponent |

**Section A** — tune a single model with t₅₀ / t₈₀ markers.
**Section B** — compare all four models side by side.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown
import ipywidgets as widgets

from fluid_bed.models.dissolution import (
    model_zero_order, model_first_order,
    model_higuchi, model_korsmeyer_peppas,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

MODEL_FNS = {
    "Zero-order":       model_zero_order,
    "First-order":      model_first_order,
    "Higuchi":          model_higuchi,
    "Korsmeyer-Peppas": model_korsmeyer_peppas,
}
COLORS = {
    "Zero-order": "royalblue", "First-order": "darkorange",
    "Higuchi": "seagreen",     "Korsmeyer-Peppas": "crimson",
}
print("Models loaded:", list(MODEL_FNS.keys()))


## Section A — Single Model Explorer

Choose one dissolution model and tune its parameters.
Grey dashed lines mark the **t₅₀** and **t₈₀** milestones (time to 50 % / 80 % release).
The Korsmeyer-Peppas **n** exponent slider is active only for that model.


In [ ]:
S = dict(continuous_update=False)


def _run04a(model="First-order", k=0.05, n=0.45, t_max_min=120.0):
    t  = np.linspace(0.01, t_max_min, 600)
    fn = MODEL_FNS[model]
    F  = fn(t, k) if model != "Korsmeyer-Peppas" else fn(t, k, n)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(t, F, lw=2.5, color=COLORS[model], label=model)

    for target, ls in [(50, "--"), (80, ":")]:
        if F[-1] >= target:
            t_m = float(np.interp(target, F, t))
            ax.axvline(t_m, color="grey", ls=ls, alpha=0.7)
            ax.axhline(target, color="grey", ls=ls, alpha=0.7)
            ax.annotate(f"t{target} = {t_m:.1f} min",
                        xy=(t_m, target), xytext=(t_m + t_max_min * 0.02, target - 8),
                        fontsize=10, color="dimgrey")

    ax.set_xlim(0, t_max_min); ax.set_ylim(0, 105)
    ax.set_xlabel("Time (min)"); ax.set_ylabel("Drug released (%)")
    kn_str = f"  n = {n:.2f}" if model == "Korsmeyer-Peppas" else ""
    ax.set_title(f"{model}   k = {k:.4f}{kn_str}")
    ax.legend(loc="lower right"); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


interact(
    _run04a,
    model     = Dropdown(options=list(MODEL_FNS), value="First-order", description="Model"),
    k         = FloatSlider(0.05,  min=0.001, max=0.50, step=0.001, description="k",
                            readout_format=".3f", **S),
    n         = FloatSlider(0.45,  min=0.10,  max=1.00, step=0.05,  description="n (K-P only)", **S),
    t_max_min = FloatSlider(120.0, min=30,    max=480,  step=30,    description="t_max (min)",  **S),
)


## Section B — All-Model Comparison

Adjust the rate constant for **each model independently** and compare their release profiles side by side.
A summary table prints t₅₀ and t₈₀ for each model.


In [ ]:
S = dict(continuous_update=False)


def _run04b(k_zero=0.50, k_first=0.05, k_higuchi=5.0,
            k_kp=0.02, n_kp=0.45, t_max_min=120.0):
    t = np.linspace(0.01, t_max_min, 600)

    curves = {
        "Zero-order":       model_zero_order(t, k_zero),
        "First-order":      model_first_order(t, k_first),
        "Higuchi":          model_higuchi(t, k_higuchi),
        "Korsmeyer-Peppas": model_korsmeyer_peppas(t, k_kp, n_kp),
    }

    fig, ax = plt.subplots(figsize=(10, 5))
    rows = []
    for name, F in curves.items():
        ax.plot(t, F, lw=2, color=COLORS[name], label=name)
        t50 = float(np.interp(50, F, t)) if F[-1] >= 50 else float("nan")
        t80 = float(np.interp(80, F, t)) if F[-1] >= 80 else float("nan")
        rows.append((name, t50, t80))

    ax.axhline(50, color="grey", ls="--", alpha=0.45)
    ax.axhline(80, color="grey", ls=":",  alpha=0.45)
    ax.text(t_max_min * 0.01, 51, "50 %", color="grey", fontsize=9)
    ax.text(t_max_min * 0.01, 81, "80 %", color="grey", fontsize=9)
    ax.set_xlim(0, t_max_min); ax.set_ylim(0, 105)
    ax.set_xlabel("Time (min)"); ax.set_ylabel("Drug released (%)")
    ax.set_title("All dissolution models — side-by-side comparison")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f"{'Model':<22} {'t50 (min)':>12} {'t80 (min)':>12}")
    print("-" * 48)
    for name, t50, t80 in rows:
        t50_s = f"{t50:8.1f}" if not np.isnan(t50) else "   > range"
        t80_s = f"{t80:8.1f}" if not np.isnan(t80) else "   > range"
        print(f"{name:<22} {t50_s:>12} {t80_s:>12}")


interact(
    _run04b,
    k_zero    = FloatSlider(0.50, min=0.01,  max=2.0,  step=0.05,  description="k zero-order",  **S),
    k_first   = FloatSlider(0.05, min=0.001, max=0.30, step=0.005, description="k first-order",
                            readout_format=".3f", **S),
    k_higuchi = FloatSlider(5.0,  min=0.5,   max=20.0, step=0.5,   description="k Higuchi",      **S),
    k_kp      = FloatSlider(0.02, min=0.001, max=0.20, step=0.005, description="k K-P",
                            readout_format=".3f", **S),
    n_kp      = FloatSlider(0.45, min=0.10,  max=1.00, step=0.05,  description="n K-P",          **S),
    t_max_min = FloatSlider(120,  min=30,    max=480,  step=30,    description="t_max (min)",    **S),
)
